# HCDE 530 — Week 5: Exploring `app_reviews_demo.csv`

This notebook follows the same **first-look** pattern as `week5_pandas_demo.ipynb`, using **`app_reviews_demo.csv`** — one row per app review (rating, text, device, dates, etc.).

Run cells top to bottom (**Shift + Enter**). Your working directory should be **`hcde530_week5`** so the CSV path resolves.

---

**Questions we answer:**

1. What does the dataset look like? (`head()`, `info()`)
2. What is the distribution of the most important column?
3. Filter to a meaningful subset — what’s in it?
4. Group by a category and take the average of a numeric column.
5. What are the missing values? Are any columns incomplete?

---

## Setup

Import **pandas** and load the CSV from this folder.

In [ ]:
import pandas as pd
import warnings

warnings.filterwarnings("ignore")
print("pandas version:", pd.__version__)

df = pd.read_csv("app_reviews_demo.csv")

---
## 1. What does your dataset look like? `head()`, `info()`

**`head()`** shows the first rows. **`info()`** lists column names, non-null counts, and dtypes.

In [ ]:
df.head()

In [ ]:
df.info()

**Readout:** Each row is one **review**: identifiers (`id`, `app`), **`category`** (type of research tool), **`rating`** (1–5), **`review`** text, **`date`**, **`helpful_votes`**, **`verified_purchase`**, **`device_type`**, **`app_version`**. Row count and dtypes come from `info()` above (expect **500** reviews in the demo file).

---
## 2. What is the distribution of your most important column?

For review data, **`rating`** is usually the key outcome: it summarizes satisfaction in one number. We use **`value_counts()`** sorted by rating so you see **how many reviews fall at each star level** (the distribution of scores).

We also show **`describe()`** on **`rating`** for mean, spread, and quartiles.

In [ ]:
# Distribution of star ratings (most important numeric outcome for this dataset)
df["rating"].value_counts().sort_index()

In [ ]:
df["rating"].describe()

---
## 3. Filter to a meaningful subset — what’s in it?

**Meaningful subset:** reviews with **`rating <= 2`** — critical feedback worth investigating separately from neutral or positive reviews.

We store the result in **`subset`** and inspect size plus a sample.

In [ ]:
subset = df[df["rating"] <= 2].copy()
print(f"Subset size: {len(subset)} rows (from {len(df)} total)")
subset.head(10)

**What’s in it:** Only **low-star** reviews (1–2). Use this slice to read themes in dissatisfied feedback or compare **`helpful_votes`** / **`device_type`** completeness vs the full table.

---
## 4. Group by a category and find the average of a numeric column

We **`groupby('category')`** (research tool type) and take **`mean()`** on **`rating`** so we can compare **average star rating** across product categories in this dataset.

In [ ]:
df.groupby("category", as_index=False)["rating"].mean().round(3).sort_values("rating")

---
## 5. What are the missing values? Are any columns incomplete?

**`isnull().sum()`** counts **NaN** per column. Text columns sometimes use **empty strings** instead of NaN; the second cell checks those.

In [ ]:
df.isnull().sum()

In [ ]:
# Empty strings in object columns (may not show up in isnull)
for col in df.select_dtypes(include="object").columns:
    blank = (df[col].astype(str).str.strip() == "").sum()
    if blank:
        print(col, "blank or whitespace-only:", blank)

**Readout for this demo file:** Expect **`device_type`** ~63 **NaN** and **`app_version`** ~111 **NaN** (versions not recorded for many rows). Core fields like **`rating`**, **`review`**, **`app`**, and **`date`** are complete. Any remaining blank-like strings in object columns show up in the loop above.

For analysis, you may later **`fillna`**, drop rows, or map missing **`device_type`** to `\"Unknown\"` — see `week5_pandas_demo.ipynb` for patterns.